# Step 2 — Feature engineering, SMOTE and XGBoost training
Input: `datasets/processed/merged_dataset.csv` (from the merge step).
Output: `model_checkpoints/xgb_pipeline.pkl` — the file your Flask `/predict` will load.

In [ ]:
# Cell 0 - environment and paths (works in Colab and locally)
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/incident-risk-predictor-files'
    !pip install -q xgboost imbalanced-learn shap
else:
    # Drive for desktop path; override with the IRP_BASE environment variable if yours differs
    BASE = os.environ.get('IRP_BASE', r'G:\My Drive\incident-risk-predictor-files')

for sub in ['datasets/raw', 'datasets/processed', 'model_checkpoints', 'reports', 'logs']:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)
print('IN_COLAB =', IN_COLAB, '| BASE =', BASE)

In [ ]:
# Cell 1 — Imports and load the merged dataset

import pandas as pd, numpy as np, joblib, json
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import sklearn, xgboost, imblearn

df = pd.read_csv(f'{BASE}/datasets/processed/merged_dataset.csv')
print(df.shape)
print(df['risk_level'].value_counts())

In [ ]:
# Cell 2 — Leakage check: columns that give the answer away
# incident_type for maintenance rows IS the Failure Type we built risk_level from,
# operational_impact came from Target (failure yes/no), and source tells the model which dataset a row is from.
# If we keep them the model scores ~100% without learning anything useful.
for col in ['incident_type', 'operational_impact', 'source']:
    print(f'\n=== {col} vs risk_level ===')
    print(pd.crosstab(df[col], df['risk_level']).head(10))

In [ ]:
# Cell 3 — Choose features and encode the target (department removed)
LEAKY = ['incident_type', 'operational_impact', 'source', 'department']
CATEGORICAL = ['shift_type', 'task_status']
NUMERIC = ['severity_score', 'tool_wear_min', 'air_temp_k', 'process_temp_k',
           'rotational_speed', 'resolution_time']
FEATURES = CATEGORICAL + NUMERIC

RISK_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
LABELS = ['Low', 'Medium', 'High']

X = df[FEATURES].copy()
y = df['risk_level'].map(RISK_MAP)
print('Dropped        :', LEAKY)
print('Features used  :', FEATURES)
print('Missing values :', X.isnull().sum().sum())

In [ ]:
# Cell 4 — Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train class counts:', y_train.value_counts().sort_index().to_dict())

In [ ]:
# Cell 5 — Pipeline: encode -> SMOTE -> XGBoost
# SMOTE sits inside the pipeline so it only ever touches training folds, never the test set.
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), CATEGORICAL),
    ('num', StandardScaler(), NUMERIC),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        objective='multi:softprob', num_class=3,
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=42, n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f'5-fold CV macro F1: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')

pipeline.fit(X_train, y_train)
print('XGBoost pipeline trained ✓')

In [ ]:
# Cell 6 — Evaluate on the held-out test set
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

print(classification_report(y_test, y_pred, target_names=LABELS, digits=3))
print(f"Macro F1      : {f1_score(y_test, y_pred, average='macro'):.3f}")
print(f"ROC-AUC (OvR) : {roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro'):.3f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('XGBoost — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{BASE}/reports/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Cell 7 — Baseline comparison (Random Forest) for your report
baseline = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
])
baseline.fit(X_train, y_train)
rf_pred = baseline.predict(X_test)
rf_proba = baseline.predict_proba(X_test)

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest'],
    'Macro F1': [f1_score(y_test, y_pred, average='macro'), f1_score(y_test, rf_pred, average='macro')],
    'ROC-AUC': [roc_auc_score(y_test, y_proba, multi_class='ovr'),
                roc_auc_score(y_test, rf_proba, multi_class='ovr')],
}).round(3)
print(comparison)
comparison.to_csv(f'{BASE}/reports/model_comparison.csv', index=False)

In [ ]:
# Cell 8 — Feature importance (quick look; SHAP comes next)
feat_names = pipeline.named_steps['prep'].get_feature_names_out()
imp = pd.Series(pipeline.named_steps['model'].feature_importances_, index=feat_names)
imp.sort_values().tail(15).plot(kind='barh', figsize=(8, 6), title='XGBoost — Top 15 features')
plt.tight_layout()
plt.savefig(f'{BASE}/reports/feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Cell 9 — Save the trained pipeline + metadata for Flask
joblib.dump(pipeline, f'{BASE}/model_checkpoints/xgb_pipeline.pkl')
# Portable copy of the XGBoost model: a booster pickled here (Linux) cannot be
# unpickled on Windows, so Flask loads the model from this file instead.
pipeline.named_steps['model'].save_model(f'{BASE}/model_checkpoints/xgb_model.ubj')

meta = {
    'features': FEATURES, 'categorical': CATEGORICAL, 'numeric': NUMERIC,
    'labels': LABELS,
    'versions': {'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__,
                 'imblearn': imblearn.__version__, 'pandas': pd.__version__},
}
with open(f'{BASE}/model_checkpoints/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Saved xgb_pipeline.pkl, xgb_model.ubj and model_meta.json to Drive')
print(json.dumps(meta['versions'], indent=2))

In [ ]:
# Cell 10 — Sanity check: reload and predict one incident, exactly as Flask will
loaded = joblib.load(f'{BASE}/model_checkpoints/xgb_pipeline.pkl')
sample = pd.DataFrame([{
    'shift_type': 'Night', 'task_status': 'Pending',
    'severity_score': 0.8, 'tool_wear_min': 210, 'air_temp_k': 302.1,
    'process_temp_k': 311.4, 'rotational_speed': 1380, 'resolution_time': 240,
}])
proba = loaded.predict_proba(sample)[0]
print('Predicted risk:', LABELS[proba.argmax()])
print({l: round(float(p), 3) for l, p in zip(LABELS, proba)})